# Notebook 13 — Data Association and Gating

This notebook builds intuition for **data association** in EKF-SLAM, with an emphasis on **nearest-neighbor matching** and **Mahalanobis gating**.

We keep everything reproducible (fixed seeds) and lightweight so the notebook runs quickly top-to-bottom.


## Learning objectives

1. Explain why data association is critical for SLAM correctness.
2. Implement nearest-neighbor association using Mahalanobis distance.
3. Choose a chi-square gating threshold from a target confidence / false-positive rate.
4. Observe failure modes such as **perceptual aliasing** (two close landmarks).
5. Connect all of this to `kiss_slam.data_association` and EKF innovation covariance `S` from the update step.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2

from kiss_slam.data_association import NearestNeighborAssociator
from kiss_slam.math_utils import mahalanobis_distance, wrap_angle
from kiss_slam.models.measurement import RangeBearingMeasurementModel
from kiss_slam.ekf_slam import EKFSLAM
from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.types import Measurement

np.set_printoptions(precision=4, suppress=True)
SEED = 13
rng = np.random.default_rng(SEED)
print(f"Seed: {SEED}")


## 1) Why data association matters

In EKF-SLAM, each measurement update assumes we know **which landmark** produced the measurement.

If association is wrong:
- the innovation is computed against the wrong predicted measurement,
- the Kalman update pushes robot/map estimates in the wrong direction,
- repeated errors can distort the map and make localization diverge.

So association is not just a bookkeeping detail — it is part of filter stability.


## 2) Nearest-neighbor association with Mahalanobis distance

For a candidate landmark, define innovation
\[
\nu = z - \hat z, \quad \nu_\phi = \mathrm{wrap}(\nu_\phi)
\]
and innovation covariance
\[
S = H\Sigma H^T + R.
\]

Use squared Mahalanobis distance
\[
d^2 = \nu^T S^{-1}\nu.
\]

Nearest-neighbor picks the landmark with smallest \(d^2\), then applies a gate threshold.


In [ ]:
def nearest_neighbor_with_gate(z, z_hats, S_list, gate_threshold):
    d2 = np.array([mahalanobis_distance(z - z_hat, S) for z_hat, S in zip(z_hats, S_list)], dtype=float)
    best = int(np.argmin(d2))
    return (best if d2[best] <= gate_threshold else None), d2

# Tiny deterministic toy example
z = np.array([8.0, np.deg2rad(4.0)])
z_hats = [np.array([7.7, np.deg2rad(2.0)]), np.array([8.4, np.deg2rad(8.0)])]
S_list = [np.diag([0.3**2, np.deg2rad(3.0)**2]), np.diag([0.3**2, np.deg2rad(3.0)**2])]

best, d2 = nearest_neighbor_with_gate(z, z_hats, S_list, gate_threshold=5.99)
print("Mahalanobis d^2 to candidates:", d2)
print("Associated index:", best)


## 3) Chi-square gating threshold (2D measurement)

For range-bearing observations, innovation is 2D, so under ideal assumptions:
\[
d^2 \sim \chi^2_2.
\]

Therefore gate thresholds are chi-square quantiles for 2 DoF.


In [ ]:
confidences = [0.90, 0.95, 0.975, 0.99, 0.999]
for c in confidences:
    print(f"P={c:>6.3f} -> gate={chi2.ppf(c, df=2):.3f}")

print("\nCommon default in this repo: 5.99 (~95% for df=2)")


## 4) Interactive simulation: two close landmarks and association flips

We simulate a robot observing two nearby landmarks. Because their expected measurements are very similar, noise can make nearest-neighbor switch between them (perceptual aliasing).

Try changing:
- `separation_m`
- `sigma_r`, `sigma_b_deg`
- `num_trials`

and rerun the cell.


In [ ]:
def simulate_association_flips(
    separation_m=0.6,
    sigma_r=0.15,
    sigma_b_deg=2.0,
    num_trials=300,
    gate=9.21,
    seed=0,
):
    rng_local = np.random.default_rng(seed)
    model = RangeBearingMeasurementModel()

    robot = np.array([0.0, 0.0, 0.0])
    landmarks = {
        0: np.array([6.0, -separation_m / 2.0]),
        1: np.array([6.0, +separation_m / 2.0]),
    }

    true_id = 0
    z_true = model.predict(robot, landmarks[true_id])
    R = np.diag([sigma_r**2, np.deg2rad(sigma_b_deg) ** 2])

    accepted_ids = []
    for _ in range(num_trials):
        noise = np.array([rng_local.normal(0.0, sigma_r), rng_local.normal(0.0, np.deg2rad(sigma_b_deg))])
        z = z_true + noise
        z[1] = wrap_angle(z[1])

        d2 = {}
        for lid, lm in landmarks.items():
            z_hat = model.predict(robot, lm)
            innovation = z - z_hat
            innovation[1] = wrap_angle(innovation[1])
            d2[lid] = mahalanobis_distance(innovation, R)

        best_id = min(d2, key=d2.get)
        accepted_ids.append(best_id if d2[best_id] <= gate else None)

    accepted = [a for a in accepted_ids if a is not None]
    flip_rate = np.nan if not accepted else np.mean(np.array(accepted) != true_id)
    return accepted_ids, flip_rate

accepted_ids, flip_rate = simulate_association_flips(
    separation_m=0.45,
    sigma_r=0.15,
    sigma_b_deg=2.5,
    num_trials=250,
    gate=5.99,
    seed=SEED,
)

vals = np.array([-1 if v is None else v for v in accepted_ids])
plt.figure(figsize=(9, 2.8))
plt.plot(vals, lw=1)
plt.yticks([-1, 0, 1], ["rejected", "landmark 0", "landmark 1"])
plt.xlabel("trial")
plt.title(f"Association outcomes; flip rate among accepted = {flip_rate:.2%}")
plt.grid(alpha=0.3)
plt.show()


## 5) Interactive simulation: gating threshold vs false matches

We compare two classes of innovations:
- **true match** innovations: \(\nu \sim \mathcal N(0, S)\)
- **false match** innovations: offset innovations (wrong landmark)

We sweep gating threshold and measure acceptance rates.


In [ ]:
def gating_tradeoff(num_samples=4000, offset=np.array([0.8, np.deg2rad(10.0)]), seed=0):
    rng_local = np.random.default_rng(seed)
    S = np.diag([0.2**2, np.deg2rad(3.0)**2])

    true_innov = rng_local.multivariate_normal(mean=np.zeros(2), cov=S, size=num_samples)
    false_innov = rng_local.multivariate_normal(mean=offset, cov=S, size=num_samples)

    d2_true = np.array([mahalanobis_distance(v, S) for v in true_innov])
    d2_false = np.array([mahalanobis_distance(v, S) for v in false_innov])

    thresholds = np.linspace(0.1, 20.0, 150)
    tpr = np.array([(d2_true <= t).mean() for t in thresholds])
    fpr = np.array([(d2_false <= t).mean() for t in thresholds])
    return thresholds, tpr, fpr

thresholds, tpr, fpr = gating_tradeoff(seed=SEED)

plt.figure(figsize=(7.5, 4))
plt.plot(thresholds, tpr, label="true-match acceptance", lw=2)
plt.plot(thresholds, fpr, label="false-match acceptance", lw=2)
plt.axvline(5.99, color="k", ls="--", alpha=0.6, label="5.99 (95% df=2)")
plt.xlabel("Gate threshold on Mahalanobis d^2")
plt.ylabel("Rate")
plt.ylim(-0.02, 1.02)
plt.grid(alpha=0.3)
plt.legend()
plt.title("Gating tradeoff")
plt.show()


## 6) Connection to `kiss_slam.data_association`

`NearestNeighborAssociator` in this repo computes exactly this pattern:
1. Predict measurement for each landmark.
2. Compute wrapped innovation and Mahalanobis distance.
3. Pick minimum-distance landmark.
4. Reject if distance exceeds gate.

Let's call it directly in a controlled toy setup.


In [ ]:
model = RangeBearingMeasurementModel()
assoc = NearestNeighborAssociator(gate_threshold=5.99)

robot_state = np.array([0.0, 0.0, 0.0])
landmark_states = {
    10: np.array([5.0, -0.2]),
    11: np.array([5.0, +0.2]),
}

z_true = model.predict(robot_state, landmark_states[10])
z_noisy = z_true + np.array([0.03, np.deg2rad(1.0)])
measurement = Measurement(landmark_id=None, range_m=float(z_noisy[0]), bearing_rad=float(z_noisy[1]))

R = np.diag([0.15**2, np.deg2rad(2.0)**2])
def innovation_cov_fn(_landmark_id: int) -> np.ndarray:
    return R

result = assoc.associate(
    measurements=[measurement],
    robot_state=robot_state,
    landmark_states=landmark_states,
    innovation_cov_fn=innovation_cov_fn,
    measurement_model=model,
)
print(result[0])


## 7) Connection to EKF innovation and `S` matrix from update

In EKF-SLAM update (for one landmark):
\[
S = H\Sigma H^T + R
\]
where `S` is the innovation covariance used both by Kalman gain and Mahalanobis gating / NIS.

The class `EKFSLAM` exposes this internally via `_innovation_covariance(...)`. We'll compute it for an initialized landmark.


In [ ]:
slam = EKFSLAM(
    motion_model=DifferentialDriveMotionModel(),
    measurement_model=RangeBearingMeasurementModel(),
)

init_meas = Measurement(landmark_id=42, range_m=4.0, bearing_rad=np.deg2rad(15.0))
slam.update([init_meas])

S = slam._innovation_covariance(42, slam.R)
print("Innovation covariance S:\n", S)

robot = slam.robot_pose()
landmark = slam.landmark_states()[42]
z_hat = slam.measurement_model.predict(robot, landmark)
z = z_hat + np.array([0.1, np.deg2rad(2.0)])
innovation = z - z_hat
innovation[1] = wrap_angle(innovation[1])
print("d^2 =", mahalanobis_distance(innovation, S))


## Exercises

### Exercise 1 — implement `mahalanobis_distance(r, S)`

Write a function that returns \(r^T S^{-1} r\). Use `np.linalg.solve` instead of explicitly inverting `S`.


In [ ]:
# TODO: implement

def mahalanobis_distance_student(r: np.ndarray, S: np.ndarray) -> float:
    raise NotImplementedError

r_test = np.array([0.3, -0.1])
S_test = np.array([[0.2, 0.01], [0.01, 0.05]])
print("Expected (reference):", mahalanobis_distance(r_test, S_test))


<details>
<summary>Solution (click to expand)</summary>

```python
def mahalanobis_distance_student(r: np.ndarray, S: np.ndarray) -> float:
    return float(r.T @ np.linalg.solve(S, r))
```

</details>

### Exercise 2 — tune a gating threshold for a given false-positive rate

Given false-match innovations from a validation run, choose a threshold such that `P(false accept) <= alpha`.

Hint: if assumptions hold, `threshold = chi2.ppf(1 - alpha, df=2)` is a strong starting point, then validate empirically.


In [ ]:
alpha = 0.05
threshold_guess = chi2.ppf(1 - alpha, df=2)
print("Initial threshold guess from chi-square:", threshold_guess)

thresholds, _, fpr = gating_tradeoff(seed=SEED)
idx = np.argmin(np.abs(thresholds - threshold_guess))
print(f"Empirical false acceptance near threshold={thresholds[idx]:.2f}: {fpr[idx]:.3f}")


## Recap

- Data association quality strongly affects EKF-SLAM stability.
- Nearest-neighbor + Mahalanobis distance is simple and effective for many cases.
- Chi-square gates tie threshold selection to a probabilistic interpretation.
- Perceptual aliasing (similar landmarks) can still cause association flips.
- In EKF update, innovation covariance `S` is the shared object behind Kalman gain, NIS, and gating.
